# A3C from Scratch: Asynchronous Advantage Actor-Critic

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/reinforcement-learning/a3c_cartpole.ipynb)

We build A3C one idea at a time on CartPole (capped at 200 steps):

1. **REINFORCE** - learn from the full Monte-Carlo return, no critic. High variance.
2. **A2C** - add a value head (critic) and learn from the *n*-step advantage. One learner.
3. **A3C** - run several workers asynchronously, each pushing gradients to a shared network.

All three balance the pole. The point is *how*: the critic cuts the variance of the
return estimate about in half, and the asynchronous workers give the steadiest training.

Companion post: **A3C from Scratch: Asynchronous Advantage Actor-Critic** on sesen.ai.

## Setup

In [ ]:
!pip install -q gymnasium torch matplotlib imageio

In [ ]:
import threading, time, numpy as np, torch, torch.nn as nn
from torch.distributions import Categorical
import gymnasium as gym
import matplotlib.pyplot as plt
torch.set_num_threads(1)

CAP, SOLVE, N_STEP = 200, 195.0, 5
def make(): return gym.make("CartPole-v1", max_episode_steps=CAP)

## The actor-critic network

Two small heads read the same 4-number CartPole observation: an **actor** that outputs
action logits `pi(a|s)`, and a **critic** that outputs a single state value `V(s)`.

In [ ]:
class ActorCritic(nn.Module):
    def __init__(self, obs=4, acts=2, h=128):
        super().__init__()
        self.pi = nn.Sequential(nn.Linear(obs, h), nn.Tanh(), nn.Linear(h, acts))  # actor
        self.v  = nn.Sequential(nn.Linear(obs, h), nn.Tanh(), nn.Linear(h, 1))     # critic
    def forward(self, x):
        return self.pi(x), self.v(x)

def evaluate(net, policy_only=False, ep=20, seed=999):
    e = make(); tot = []
    for k in range(ep):
        s, _ = e.reset(seed=seed + k); done = False; R = 0
        while not done:
            with torch.no_grad():
                out = net(torch.as_tensor(s, dtype=torch.float32))
                logits = out if policy_only else out[0]
            s, r, term, trunc, _ = e.step(int(logits.argmax())); R += r; done = term or trunc
        tot.append(R)
    return float(np.mean(tot))

## REINFORCE: Monte-Carlo returns, no critic

The high-variance baseline. Collect a batch of full episodes, compute the discounted
return for each step, normalise, and do gradient ascent on `log pi(a|s) * G`.

In [ ]:
class PolicyNet(nn.Module):
    def __init__(self, obs=4, acts=2, h=128):
        super().__init__(); self.f = nn.Sequential(nn.Linear(obs, h), nn.Tanh(), nn.Linear(h, acts))
    def forward(self, x): return self.f(x)

def train_reinforce(gamma=0.99, lr=1e-2, max_frames=100_000, seed=0, batch_eps=5):
    torch.manual_seed(seed); np.random.seed(seed)
    env = make(); net = PolicyNet(); opt = torch.optim.Adam(net.parameters(), lr=lr)
    frames = 0; curve = []; nxt = 5000
    while frames < max_frames:
        S, A, R = [], [], []
        for _ in range(batch_eps):
            s, _ = env.reset(); rs, ss, aa = [], [], []; done = False
            while not done:
                with torch.no_grad(): logits = net(torch.as_tensor(s, dtype=torch.float32))
                a = int(Categorical(logits=logits).sample())
                s2, r, term, trunc, _ = env.step(a); done = term or trunc
                ss.append(s); aa.append(a); rs.append(r); s = s2; frames += 1
            G = 0.0; disc = []
            for r in reversed(rs): G = r + gamma * G; disc.append(G)
            disc.reverse(); S += ss; A += aa; R += disc
        St = torch.as_tensor(np.array(S), dtype=torch.float32); At = torch.as_tensor(np.array(A))
        Rt = torch.as_tensor(np.array(R), dtype=torch.float32); Rt = (Rt - Rt.mean()) / (Rt.std() + 1e-8)
        loss = -(Categorical(logits=net(St)).log_prob(At) * Rt).mean()
        opt.zero_grad(); loss.backward(); opt.step()
        if frames >= nxt:
            curve.append((frames, evaluate(net, policy_only=True))); nxt += 5000
    return curve, net

## A2C: one learner, n-step advantage

Add the critic. After every `N_STEP` steps, form the n-step return, subtract the critic
to get the advantage `A = R - V(s)`, and minimise **critic loss + actor loss** together.

In [ ]:
def train_a2c(gamma=0.9, lr=3e-4, max_frames=100_000, seed=0):
    torch.manual_seed(seed); np.random.seed(seed)
    env = make(); net = ActorCritic(); opt = torch.optim.Adam(net.parameters(), lr=lr, betas=(0.92, 0.999))
    s, _ = env.reset(seed=seed); frames = 0; curve = []; nxt = 5000
    bs, ba, br = [], [], []
    while frames < max_frames:
        with torch.no_grad(): logits, _ = net(torch.as_tensor(s, dtype=torch.float32))
        a = int(Categorical(logits=logits).sample())
        s2, r, term, trunc, _ = env.step(a); done = term or trunc
        bs.append(s); ba.append(a); br.append(r); s = s2; frames += 1
        if len(bs) >= N_STEP or done:
            R = 0.0 if done else float(net(torch.as_tensor(s2, dtype=torch.float32))[1])
            tgt = []
            for rr in reversed(br): R = rr + gamma * R; tgt.append(R)
            tgt.reverse()
            S = torch.as_tensor(np.array(bs), dtype=torch.float32); A = torch.as_tensor(np.array(ba))
            Rt = torch.as_tensor(np.array(tgt), dtype=torch.float32)
            logits, V = net(S); V = V.squeeze(1); adv = Rt - V
            m = Categorical(logits=logits)
            loss = adv.pow(2).mean() - (m.log_prob(A) * adv.detach()).mean()
            opt.zero_grad(); loss.backward(); opt.step(); bs, ba, br = [], [], []
        if done: s, _ = env.reset()
        if frames >= nxt: curve.append((frames, evaluate(net))); nxt += 5000
    return curve, net

## A3C: asynchronous workers

Each worker keeps its own environment and a local copy of the network. It rolls `N_STEP`
steps, computes the same advantage update, and applies the gradient to a **shared** global
network under a lock. Four workers train one network in parallel, with no replay buffer.
Because there are ~4x as many updates per frame, we use a smaller learning rate.

In [ ]:
def train_a3c(workers=4, gamma=0.9, lr=1e-4, max_frames=120_000, seed=0):
    torch.manual_seed(seed); np.random.seed(seed)
    gnet = ActorCritic(); opt = torch.optim.Adam(gnet.parameters(), lr=lr, betas=(0.92, 0.999))
    lock = threading.Lock(); counter = [0]; stop = [False]

    def worker():
        env = make(); local = ActorCritic(); s, _ = env.reset(); bs, ba, br = [], [], []
        while not stop[0]:
            local.load_state_dict(gnet.state_dict()); done = False
            for _ in range(N_STEP):
                with torch.no_grad(): logits, _ = local(torch.as_tensor(s, dtype=torch.float32))
                a = int(Categorical(logits=logits).sample())
                s2, r, term, trunc, _ = env.step(a); done = term or trunc
                bs.append(s); ba.append(a); br.append(r); s = s2
                if done: break
            with lock:
                counter[0] += len(br)
                R = 0.0 if done else float(local(torch.as_tensor(s, dtype=torch.float32))[1])
                tgt = []
                for rr in reversed(br): R = rr + gamma * R; tgt.append(R)
                tgt.reverse()
                S = torch.as_tensor(np.array(bs), dtype=torch.float32); A = torch.as_tensor(np.array(ba))
                Rt = torch.as_tensor(np.array(tgt), dtype=torch.float32)
                logits, V = local(S); V = V.squeeze(1); adv = Rt - V
                m = Categorical(logits=logits)
                loss = adv.pow(2).mean() - (m.log_prob(A) * adv.detach()).mean()
                opt.zero_grad(); local.zero_grad(); loss.backward()
                for gp, lp in zip(gnet.parameters(), local.parameters()): gp.grad = lp.grad
                opt.step()
            bs, ba, br = [], [], []
            if done: s, _ = env.reset()

    ts = [threading.Thread(target=worker) for _ in range(workers)]
    for t in ts: t.start()
    curve = []; nxt = 5000; solved = None
    while counter[0] < max_frames:
        if counter[0] >= nxt:
            er = evaluate(gnet); curve.append((counter[0], er)); nxt += 5000
            if er >= SOLVE and solved is None: solved = counter[0]
        time.sleep(0.02)
    stop[0] = True
    for t in ts: t.join()
    print(f"A3C solved at {solved} frames" if solved else "A3C did not reach the bar")
    return curve, gnet

## Train and compare

One run of each (a few minutes on a CPU). All three should climb to the solved line at 195.

In [ ]:
t0 = time.time()
c_rf, _        = train_reinforce(seed=0)
c_a2c, _       = train_a2c(seed=0)
c_a3c, net_a3c = train_a3c(seed=0)
print(f"trained in {time.time()-t0:.0f}s")

plt.figure(figsize=(8, 5))
for curve, lab, col in [(c_rf, "REINFORCE (no critic)", "#94a3b8"),
                        (c_a2c, "A2C (1 learner)", "#2563eb"),
                        (c_a3c, "A3C (4 workers)", "#dc2626")]:
    x, y = zip(*curve); plt.plot(np.array(x)/1000, y, label=lab, color=col, lw=2)
plt.axhline(SOLVE, ls="--", color="gray"); plt.xlabel("frames (thousands)")
plt.ylabel("evaluation reward"); plt.legend(); plt.title("All three learn to balance CartPole"); plt.show()

## The critic's payoff: variance versus n

Why does the critic help? It lets us stop trusting real reward after `n` steps. Measure the
variance of the return estimate on a fixed policy as we vary `n`: more bootstrapping (smaller
`n`) means lower variance. The Monte-Carlo return REINFORCE uses (`n = infinity`) is the noisiest.

In [ ]:
def nstep_return(rewards, boot, n, gamma):
    g = sum((gamma ** k) * rewards[k] for k in range(min(n, len(rewards))))
    if n < len(rewards): g += (gamma ** n) * boot
    return g

def return_variance(net, ns_list, gamma=0.9, n_anchors=30, n_rollouts=60, seed=0):
    torch.manual_seed(seed); rng = np.random.RandomState(seed); env = make()
    anchors = []; s, _ = env.reset(seed=seed)
    for _ in range(2000):
        with torch.no_grad(): logits, _ = net(torch.as_tensor(s, dtype=torch.float32))
        s, r, te, tr, _ = env.step(int(logits.argmax()))
        if len(anchors) < n_anchors and rng.rand() < 0.15: anchors.append(s.copy())
        if te or tr: s, _ = env.reset()
        if len(anchors) >= n_anchors: break
    out = {n: [] for n in ns_list}
    for a_state in anchors:
        rolls = []
        for _ in range(n_rollouts):
            env.reset(seed=int(rng.randint(1 << 30))); env.unwrapped.state = np.array(a_state, float)
            s = np.array(a_state, np.float32); rs, bv = [], []
            for _t in range(CAP):
                with torch.no_grad(): logits, _ = net(torch.as_tensor(s, dtype=torch.float32))
                a = int(Categorical(logits=logits).sample()); s2, r, te, tr, _ = env.step(a); rs.append(r)
                with torch.no_grad(): bv.append(float(net(torch.as_tensor(s2, dtype=torch.float32))[1]))
                s = np.asarray(s2, np.float32)
                if te or tr: break
            rolls.append((rs, bv))
        for n in ns_list:
            est = [nstep_return(rs, (bv[n-1] if n <= len(bv) else 0.0), n, gamma) for rs, bv in rolls]
            out[n].append(np.var(est))
    return {n: float(np.mean(v)) for n, v in out.items()}

# use a partly-trained policy (variable episode lengths -> visible variance)
_, midnet = train_a2c(seed=1, max_frames=9000)
ns_list = [1, 2, 3, 5, 10, 20, CAP]
var = return_variance(midnet, ns_list)
plt.figure(figsize=(7.5, 4.5))
plt.plot(range(len(ns_list)), [var[n] for n in ns_list], "o-", color="#dc2626", lw=2)
plt.xticks(range(len(ns_list)), [str(n) for n in ns_list[:-1]] + ["MC"])
plt.xlabel("n (steps before trusting the critic)"); plt.ylabel("variance of the return estimate")
plt.title("More bootstrapping, less variance"); plt.show()
print({n: round(var[n], 3) for n in ns_list})

## Watch it balance

In [ ]:
import imageio
env = gym.make("CartPole-v1", max_episode_steps=CAP, render_mode="rgb_array")
frames = []; s, _ = env.reset(seed=7)
for _ in range(CAP):
    with torch.no_grad(): logits, _ = net_a3c(torch.as_tensor(s, dtype=torch.float32))
    frames.append(env.render()); s, r, te, tr, _ = env.step(int(logits.argmax()))
    if te or tr: break
imageio.mimsave("cartpole.gif", frames[::2], fps=30)
print(f"balanced {len(frames)} steps -> cartpole.gif")

## Takeaways

- **The critic reduces variance.** The advantage `A = R - V(s)` subtracts a state-dependent
  baseline that leaves the policy gradient unbiased but far less noisy.
- **N-step returns trade bias for variance.** Small `n` leans on the critic; the Monte-Carlo
  return REINFORCE uses is the highest-variance extreme.
- **A3C parallelises without a replay buffer.** Asynchronous workers decorrelate on-policy data,
  giving stable training. OpenAI later showed a synchronous version (A2C) works just as well.
- **This is the template.** PPO is this advantage objective plus a clipped, safe policy update.